# 第 12 章　机器学习进阶
## 特征工程、样本不均衡与 Stacking

::: {.callout-note}
## 本章要点

1. **金融特征工程**：时序滚动特征、技术指标、因子构造
2. **样本不均衡处理**：SMOTE、类权重、阈值调整
3. **Stacking / Blending**：用元学习器组合多个基模型
4. **综合案例**：新股破发预测（IPO 首日收盘价 < 发行价）

DDML 的完整实现已在第 8 章介绍，本章聚焦预测任务的工程细节。
:::


In [ ]:
# ── 第 12 章　ML 进阶　──────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb, lightgbm as lgb

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
OUTPUT = 'output'; os.makedirs(OUTPUT, exist_ok=True)
RNG = np.random.default_rng(42)
print('环境就绪 ✓')


---

## 1　金融时序特征工程

金融机器学习的特征构造遵循几个原则：

1. **不使用未来数据**（前视偏差，见第 10 章）
2. **滚动窗口统计量**：用过去 $N$ 期的均值、标准差、动量
3. **技术指标**：MA、RSI、MACD 等（本质是滚动统计量的组合）
4. **截面排序**：将原始值替换为在截面分位数排名（减少量纲影响）


In [ ]:
# ── 1.1  时序滚动特征构造 ────────────────────────────────────────────
# 模拟一只股票的日度收益率数据
T_feat = 500
rets   = RNG.normal(0.0003, 0.012, T_feat)
price  = 100 * np.cumprod(1 + rets)
vol    = np.abs(rets) * 100

df_feat = pd.DataFrame({'ret': rets, 'price': price, 'vol': vol})

# 滚动特征（注意 shift(1) 避免前视偏差）
for window in [5, 20, 60]:
    df_feat[f'ret_mean_{window}d'] = df_feat['ret'].shift(1).rolling(window).mean()
    df_feat[f'ret_std_{window}d']  = df_feat['ret'].shift(1).rolling(window).std()
    df_feat[f'mom_{window}d']      = df_feat['price'].shift(1).pct_change(window)

# RSI（14 日）
delta    = df_feat['ret'].shift(1)
gain     = delta.clip(lower=0).rolling(14).mean()
loss     = (-delta.clip(upper=0)).rolling(14).mean()
df_feat['RSI_14'] = 100 - 100 / (1 + gain / (loss + 1e-8))

print(f'原始特征数：3  →  构造后特征数：{df_feat.shape[1]}')
print(df_feat.dropna().describe().round(4).to_string())


---

## 2　样本不均衡处理

金融任务普遍存在样本不均衡：违约率 2%、破发率 10%……
直接训练会让模型偏向多数类（预测所有样本为「不违约」，准确率 98%，毫无用处）。

**三种主流策略：**

| 策略 | 方法 | 适用场景 |
|------|------|----------|
| **类权重** | `class_weight='balanced'`（自动）| 简单有效，首选尝试 |
| **欠采样** | 随机删除多数类样本 | 数据量充足时 |
| **过采样** | SMOTE 合成少数类样本 | 少数类太少时（< 5%）|


In [ ]:
# ── 2.1  IPO 破发预测数据 ────────────────────────────────────────────
N_IPO = 2000
BREAK_RATE = 0.12   # 破发率约 12%

X_ipo = RNG.normal(0, 1, (N_IPO, 10))
ipo_feat = ['PE_ratio','PB_ratio','Market_size','Underwriter_rep',
            'Lock_period','Oversubscription','IPO_size','Industry_hot',
            'Market_ret_30d','Noise']
log_odds_ipo = -2.0 + 0.8*X_ipo[:,0] - 0.5*X_ipo[:,3] - 0.6*X_ipo[:,5]
prob_break   = 1 / (1 + np.exp(-log_odds_ipo))
Y_ipo = RNG.binomial(1, prob_break)

df_ipo = pd.DataFrame(X_ipo, columns=ipo_feat)
df_ipo['break_flag'] = Y_ipo

split = int(N_IPO * 0.7)
X_tr, X_te = X_ipo[:split], X_ipo[split:]
y_tr, y_te = Y_ipo[:split], Y_ipo[split:]
print(f'IPO 数据：N={N_IPO}  破发率={Y_ipo.mean():.3f}')

# 对比：无权重 vs class_weight='balanced'
for cw, label in [(None,'无权重'), ('balanced','class_weight=balanced')]:
    m = lgb.LGBMClassifier(n_estimators=200, random_state=42,
                            class_weight=cw, verbose=-1)
    m.fit(X_tr, y_tr)
    prob = m.predict_proba(X_te)[:,1]
    print(f'{label:<30}: AUC={roc_auc_score(y_te,prob):.4f}  AP={average_precision_score(y_te,prob):.4f}')


---

## 3　Stacking：组合多个基模型

**Stacking** 用一个元学习器（meta-learner）
把多个基模型的预测值作为新特征，学习如何最优组合它们。

关键细节：基模型的训练集预测必须用**交叉验证（Out-of-Fold）**生成，
否则元学习器会过拟合基模型在训练集上的完美预测。


In [ ]:
# ── 3.1  Stacking 实现 ───────────────────────────────────────────────
base_models = [
    ('rf',   RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
    ('xgb',  xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                random_state=42, verbosity=0, eval_metric='logloss')),
    ('lgbm', lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                 random_state=42, verbose=-1)),
]
meta = LogisticRegression(C=0.5)

stacker = StackingClassifier(
    estimators=base_models, final_estimator=meta,
    cv=5, stack_method='predict_proba', passthrough=False
)
stacker.fit(X_tr, y_tr)
prob_stack = stacker.predict_proba(X_te)[:,1]

print('模型对比（IPO 破发预测）：')
for name, m in base_models:
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:,1]
    print(f'  {name:<6}: AUC={roc_auc_score(y_te,p):.4f}  AP={average_precision_score(y_te,p):.4f}')
print(f'  {'Stack':<6}: AUC={roc_auc_score(y_te,prob_stack):.4f}  AP={average_precision_score(y_te,prob_stack):.4f}')


---

## 4　章末练习

**练习 1（特征工程）**
对 A 股某板块的日度收益率数据，
构造 5/20/60 日滚动均值、标准差、动量，以及 RSI 和 MACD，
用这些特征预测下一交易日收益率的正负（二分类），
报告 AUC 和 AP。

**练习 2（SMOTE）**
对 IPO 破发数据，尝试 SMOTE 过采样（`pip install imbalanced-learn`），
比较与 `class_weight='balanced'` 方法的 AP 差异，
讨论哪种方法在本场景下更合适。

**练习 3（Stacking 设计）**
把 Logistic Regression、Random Forest、LightGBM 三个基模型
分别用于不同特征子集（Logistic 用线性特征，树模型用全部特征），
用 Ridge 作为元学习器，比较与统一特征 Stacking 的效果差异。
